# 02 · EDA del abandono: la foto del asociado

**Diseño:** se observa a cada asociado en el corte del extracto y se pregunta si se retira en los 3 meses
siguientes.

- **Población:** los asociados del extracto. En el extracto real todos están activos.
- **Objetivo:** `EVENTO_RETIRO_3M_SINTETICO`, disponible únicamente en los datos generados: el fondo confirmó que no
  se puede saber quién del corte se retiró después ([docs/datos_reales.md](../docs/datos_reales.md)). `#RETIROS` es
  historia acumulada (salidas anteriores de quienes reingresaron) y queda como predictor.
- **Variables:** las 44 columnas del contrato v2.2; `EDAD`, `ANTIGUEDAD_F_MESES` y `ANTIGUEDAD_LA_MESES` ya llegan
  calculadas al corte (`abandono.preparar`).

Con los 50 de `data/ejemplo` hay muy pocos retiros: por eso, si `DATOS_ASOCIADOS` tiene pocas filas, este notebook usa
los **mismos 5.000 de desarrollo con que se elige el campeón** (`campeon.datos_eleccion`, semilla 7, 10 % de retiro;
`make datos-desarrollo` los deja en `data/processed/`).

> **Datos sintéticos:** las tasas y las diferencias entre los que se quedan y los que se van las fija
> `generador.py`. Sirven para construir y probar el análisis, **no son evidencia sobre el fondo**.

In [ ]:
import plotly.express as px

from proyecto_ml import abandono as a
from proyecto_ml import campeon as c
from proyecto_ml.config import aviso_origen, get_settings

HORIZONTE = a.HORIZONTE_MESES  # 3 meses: el mismo de make campeon
COLORES = {"sin evento futuro": "#2a78d6", "con evento futuro": "#eb6834"}  # paleta de evaluate.py, en orden fijo

df = a.cargar(get_settings().datos_asociados)
if len(df) < 500:
    print(f"{len(df)} filas en DATOS_ASOCIADOS: muy pocas para la foto. Se usan los datos de desarrollo.")
    df, fuente = c.datos_eleccion(origen="sintetico")  # los mismos 5.000 de make campeon
    print(aviso_origen("sintetico"))
print(f"{len(df)} asociados, corte {a.FECHA_CORTE.date()}")

## 1. ¿Por qué el horizonte ya no cambia la foto?

Este notebook empezó con varias fotos (t0 = corte − 1, 3, 6 o 12 meses), quitando a quienes se habían retirado antes de
t0. Eso necesitaba una fecha de retiro, y el esquema definitivo no la trae: el fondo entrega **un solo corte** y no
puede decir quién se retiró después. Por eso `abandono.foto` conserva el lote completo y solo anota t0; la tabla
muestra que la población y la tasa son las mismas en todos los horizontes.

In [ ]:
a.resumen_foto(df).round(3)

## 2. La foto elegida: población y objetivo

In [ ]:
vista = a.foto(df, HORIZONTE)
X = a.preparar(vista)  # las variables del modelo
evento = a.objetivo_binario(vista)
clase = evento.map({0: "sin evento futuro", 1: "con evento futuro"}).rename("clase")
print(
    f"t0 = {a.fecha_observacion(HORIZONTE).date()} · {len(vista)} asociados · "
    f"{evento.sum()} tienen evento futuro sintético ({evento.mean():.1%})"
)
clase.value_counts().to_frame("asociados")

## 3. Comportamiento con el fondo: promedio por clase

Los conteos de solicitudes y productos son la señal principal del generador. El fondo confirmó que se miden **al
corte** y que `ULT3MES` son los tres meses anteriores (respuesta de 2026-11-21). `#RETIROS` se muestra aparte, como
historia.

In [ ]:
promedios = X[[*a.COMPORTAMIENTO, a.RETIROS_HISTORICOS]].groupby(clase).mean().T.rename_axis("variable").reset_index()
largo = promedios.melt(id_vars="variable", var_name="clase", value_name="promedio")
fig = px.bar(
    largo,
    y="variable",
    x="promedio",
    color="clase",
    barmode="group",
    orientation="h",
    color_discrete_map=COLORES,
    labels={"promedio": "Promedio por asociado", "variable": ""},
    title="Comportamiento promedio por clase (datos sintéticos)",
    height=520,
)
fig.update_layout(template="plotly_white", legend_title_text="")
fig

## 4. Perfil: edad, antigüedad y salario por clase

`SALARIO` sin los valores imposibles (`abandono.SALARIO_PLAUSIBLE`).

In [ ]:
for columna in ["EDAD", "ANTIGUEDAD_F_MESES", "ANTIGUEDAD_LA_MESES", "SALARIO"]:
    fig = px.box(
        X.assign(clase=clase),
        x="clase",
        y=columna,
        color="clase",
        color_discrete_map=COLORES,
        points=False,
        title=f"{columna} por clase (al corte)",
    )
    fig.update_layout(template="plotly_white", showlegend=False)
    fig.show()

## 5. Categóricas: tasa de evento por categoría

`ESTRATO` en blanco aparece como `SIN_DATO`. `CARGO` no se muestra: llega idéntico a `CODPROFESION`.

In [ ]:
for columna in ["#SOLICITUD_DISMINUCIONCUOTAS", "TIPOCONTRATO", "SEXO", "ESTRATO", "ESTUDIOS", "ESTADOCIVIL"]:
    tasa = (
        vista.assign(**{columna: X[columna], "evento_retiro": evento})
        .groupby(columna)["evento_retiro"]
        .agg(["mean", "size"])
        .reset_index()
    )
    fig = px.bar(
        tasa,
        x=columna,
        y="mean",
        hover_data={"size": True},
        labels={"mean": "Tasa de evento futuro", "size": "asociados"},
        title=f"Tasa de evento futuro por {columna} (datos sintéticos)",
        color_discrete_sequence=[COLORES["con evento futuro"]],
    )
    fig.update_yaxes(tickformat=".0%")
    fig.update_layout(template="plotly_white")
    fig.show()

## 6. Hallazgos y estado de las preguntas

- **Diseño:** una sola foto, al corte. Con el esquema definitivo no hay fecha de retiro, así que el horizonte no cambia
  la población: los 5.000 de desarrollo entran todos y el 10 % tiene evento futuro sintético. Son cifras del
  generador, no del fondo.
- **Con 50 filas no se puede:** hay muy pocos eventos. Se trabaja con los 5.000 de desarrollo.
- **Preguntas que se le hicieron al fondo** (respuestas completas en [docs/datos_reales.md](../docs/datos_reales.md)):
  1. ¿Los conteos tienen fecha? Se miden al corte; `ULT3MES` son los tres meses anteriores.
  2. ¿Tasa de retiro anual real? No está disponible y no se publicará.
  3. ¿Se puede saber quién del corte se retiró en los tres meses siguientes? No: el abandono no se entrena ni se
     evalúa con datos reales.
  4. Proporciones agregadas (códigos, conteos, empresa principal): se pueden publicar redondeadas y son las que usa
     `generador.py`.